# Assignment 2 — Facial Emotion Recognition with a Vision Transformer (happy / sad / neutral)

**What this notebook does**

| Step | Content |
|---|---|
| 1 | Audits the A1 dataset and builds a **leak-free split by source photo** (augmented variants stay with their source; validation/test use only real source photos) |
| 2 | Re-runs the **A1 baselines** (HOG + SVM, simple CNN, MobileNetV2) on the clean split, so the comparison is fair |
| 3 | Fine-tunes **ViT-B/16** (Vision Transformer) — the SOTA classifier |
| 4 | **Ablation**: ViT without augmentation / with augmentation / with augmentation + ArtAug images (if provided) |
| 5 | **Explainability**: attention-rollout heatmaps |
| 6 | **Latency** on GPU and CPU |
| 7 | **Evaluation under animation**: per-frame predictions on talking-head clips (VASA-1 concept, generated with LivePortrait/SadTalker) → emotion timeline + Δmacro-F1 vs stills |
| 8 | Exports the model and launches a **Gradio demo** (upload/webcam → emotion → heatmap → latency) |

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.

**Google Drive layout** (only the dataset folder is required; the other folders are optional and are picked up automatically when they exist):
```
MyDrive/Dataset-folder/dataset/{happy,sad,neutral}/...          <- A1 dataset (required)
MyDrive/Dataset-folder/dataset/manifest.csv                     <- optional: filename,source_id[,is_original]
MyDrive/A2/artaug/{happy,sad,neutral}/<source>__artaug_1.png    <- optional: ArtAug-enhanced TRAIN photos
MyDrive/A2/clips/{happy,sad,neutral}/<source>__clip1.mp4        <- optional: talking-head clips of TEST photos
MyDrive/A2/results/                                             <- everything this notebook saves
```
`<source>` is the `source_key` written to `train_sources.csv` / `test_sources.csv` in step 1.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 0. Settings

In [ ]:
DATASET_DIR = '/content/drive/MyDrive/Dataset-folder/dataset'
A2_DIR      = '/content/drive/MyDrive/A2'
ARTAUG_DIR  = f'{A2_DIR}/artaug'
CLIPS_DIR   = f'{A2_DIR}/clips'
RESULTS_DIR = f'{A2_DIR}/results'
MANIFEST    = f'{DATASET_DIR}/manifest.csv'

EMOTIONS = ['happy', 'sad', 'neutral']
SEED     = 42
SPLIT    = {'train': 0.70, 'val': 0.15, 'test': 0.15}

USE_ALL_PHOTOS     = True   # True = every photo is used (incl. names with _aug / (1)); variants still stay in the same split as their source
GROUP_BY_NAME      = True   # group 'x_aug05', 'x (1)', 'x_original' as one source photo 'x' (prevents train/test leakage)
NEAR_DUP_HAMMING   = 4      # photos whose 64-bit difference hash differs in <= 4 bits (also mirrored) are treated as the same shot
EVAL_ONE_PER_GROUP = False  # True = keep only one photo per source group in val/test

IMG_SIZE        = 224
BATCH           = 16
VIT_CHECKPOINT  = 'google/vit-base-patch16-224-in21k'
PRETRAINED      = True      # ImageNet weights for MobileNetV2 / ViT
VIT_EPOCHS      = 15
CNN_EPOCHS      = 30
MOBILENET_EPOCHS = (10, 10) # (head only, fine-tuning)
PATIENCE        = 5         # early stopping on validation loss
FRAME_STEP      = 3         # use every 3rd frame of each talking-head clip

## 1. Dataset audit and leak-free split

**Grouping rule** (used when `manifest.csv` is absent):
the *source key* of a file is its name with copy markers removed — `_augNN`, `_original`, `_orig` and Windows copy suffixes such as ` (1)`.
Examples: `happy_001_aug05 (1)` → `happy_001`; `neutral_team9_photo08_aug05` → `neutral_team9_photo08`; `sad_extra_001_aug11` → `sad_extra_001`.
A file is a **real source photo** only if its name has no `_augNN` marker and no ` (n)` copy suffix. Nothing else is declared an original automatically.

Both `_aug05` and `_aug_05` styles are recognised. If a name reduces to just the emotion word (e.g. `happy_aug01` … `happy_aug40` → `happy`), the files cannot be one photo and are treated as separate photos.

Then:
1. identical files (same MD5) are removed — they are the same picture; an identical file found in two emotion folders is dropped entirely (ambiguous label);
2. **near-duplicate shots** (difference hash ≤ `NEAR_DUP_HAMMING` bits, also checked against the mirrored image) inside one emotion are merged into one group;
3. groups are assigned so that **each emotion gets about 70 / 15 / 15 of its images** (seed 42; large groups are placed first, small ones fill the gaps) — a photo and all its variants always land in the **same** split, so no test photo has a copy in training.

`USE_ALL_PHOTOS = True` (default): every file is used, including `_aug` / ` (1)` files, in train, validation and test.
`USE_ALL_PHOTOS = False`: stricter mode — validation/test use only real source photos and augmentation-only groups go to training.

In [ ]:
import os, re, glob, json, time, copy, hashlib, random, shutil, platform
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt, seaborn as sns
from collections import defaultdict

random.seed(SEED); np.random.seed(SEED)
os.makedirs(RESULTS_DIR, exist_ok=True)
DATA_DIR = '/content/data_a2'
if not os.path.exists(DATA_DIR):
    shutil.copytree(DATASET_DIR, DATA_DIR)

IMG_EXTS = ('.png', '.jpg', '.jpeg', '.webp', '.bmp')
AUG_PAT  = r'[_\-\s]aug[_\-\s]?\d+'
COPY_PAT = r'\(\d+\)\s*$'
STRIP_RES = [re.compile(r'\s*\(\d+\)$'), re.compile(r'[_\-\s]aug[_\-\s]?\d+$', re.I), re.compile(r'[_\-\s](?:original|orig)$', re.I)]

def source_key(stem):
    s, changed = stem.strip().lower(), True
    while changed:
        changed = False
        for r in STRIP_RES:
            new = r.sub('', s).strip()
            if new != s:
                s, changed = new, True
    return s

def file_md5(p):
    with open(p, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

def dhash(p):
    # returns (hash, hash of the mirrored image) so horizontally flipped copies are detected too
    g = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if g is None:
        return None
    out = []
    for im in (g, cv2.flip(g, 1)):
        s = cv2.resize(im, (9, 8), interpolation=cv2.INTER_AREA)
        out.append((s[:, 1:] > s[:, :-1]).flatten())
    return out

def hamming(A, B):
    return A @ (1 - B).T + (1 - A) @ B.T

def hamming_flip(paths_a, paths_b):
    A  = np.stack([hashes[p][0] for p in paths_a]).astype(np.float32)
    B  = np.stack([hashes[p][0] for p in paths_b]).astype(np.float32)
    Bf = np.stack([hashes[p][1] for p in paths_b]).astype(np.float32)
    Af = np.stack([hashes[p][1] for p in paths_a]).astype(np.float32)
    # symmetric: a vs b, a vs mirrored b, mirrored a vs b
    return np.minimum(np.minimum(hamming(A, B), hamming(A, Bf)), hamming(Af, B))

excluded = []   # (path, label, reason)

rows = []
for e in EMOTIONS:
    for p in sorted(glob.glob(os.path.join(DATA_DIR, e, '*'))):
        if p.lower().endswith(IMG_EXTS):
            rows.append({'path': p, 'file': os.path.basename(p), 'label': e,
                         'stem': os.path.splitext(os.path.basename(p))[0]})
df = pd.DataFrame(rows)
print(f'Files found: {len(df)}')

hashes = {p: dhash(p) for p in df.path}
for p in [p for p, h in hashes.items() if h is None]:
    excluded.append((p, df.loc[df.path == p, 'label'].iat[0], 'unreadable image file'))
df = df[df.path.map(lambda p: hashes[p] is not None)].copy()
df['md5'] = df.path.map(file_md5)

# --- source groups: manifest.csv if present, otherwise the file-name rule ---
df['is_original'] = ~df.stem.str.contains(AUG_PAT, case=False) & ~df.stem.str.contains(COPY_PAT)
df['source'] = df.stem.map(source_key)
RULE = 'file-name rule'
if not GROUP_BY_NAME:
    df['source'], df['is_original'] = df.stem.str.lower(), True
    RULE = 'every file is a separate photo (GROUP_BY_NAME = False)'
if os.path.exists(MANIFEST):
    man = pd.read_csv(MANIFEST)
    man['file'] = man['filename'].astype(str).map(os.path.basename)
    cols = ['file', 'source_id'] + (['is_original'] if 'is_original' in man.columns else [])
    df = df.merge(man[cols].rename(columns={'is_original': 'is_original_m'}), on='file', how='left')
    has = df.source_id.notna()
    df.loc[has, 'source'] = df.loc[has, 'source_id'].astype(str).str.lower()
    if 'is_original_m' in df:
        df.loc[has, 'is_original'] = df.loc[has, 'is_original_m'].astype(str).str.lower().isin(['1', 'true', 'yes'])
    RULE = f'manifest.csv ({has.sum()} files), file-name rule for the remaining {(~has).sum()}'
print('Grouping rule:', RULE)

# --- exact duplicates ---
n_labels = df.groupby('md5').label.nunique()
conflict = set(n_labels[n_labels > 1].index)
for _, r in df[df.md5.isin(conflict)].iterrows():
    excluded.append((r.path, r.label, 'identical file found in different emotion folders'))
df = df[~df.md5.isin(conflict)]
df['is_copy'] = df.stem.str.contains(COPY_PAT)
df = df.sort_values(['md5', 'is_original', 'is_copy', 'file'], ascending=[True, False, True, True])
for _, r in df[df.duplicated('md5', keep='first')].iterrows():
    excluded.append((r.path, r.label, 'byte-identical duplicate of another file'))
df = df.drop_duplicates('md5', keep='first').sort_values(['label', 'file']).reset_index(drop=True)
df['group'] = df.label + '::' + df.source

# --- a name that reduces to just the emotion word is not one photo (e.g. 'happy_aug01' ... 'happy_aug40' -> 'happy') ---
too_big = sorted(set(df.group[df.source.isin(EMOTIONS + [''])]))
if len(too_big):
    m = df.group.isin(too_big)
    print(f'{m.sum()} files whose name reduces to just the emotion word are treated as separate photos:', list(too_big)[:5])
    df.loc[m, 'source'] = df.loc[m, 'stem'].str.lower()
    df.loc[m, 'is_original'] = True
    df['group'] = df.label + '::' + df.source

# --- merge near-duplicate shots within each emotion (union-find over groups) ---
parent = {g: g for g in df.group.unique()}
def find(g):
    while parent[g] != g:
        parent[g] = parent[parent[g]]
        g = parent[g]
    return g

near_pairs = []
for e in EMOTIONS:
    idx = np.where(df.label.values == e)[0]
    p_e = df.path.values[idx]
    D = hamming_flip(p_e, p_e)                              # Hamming distances (normal and mirrored)
    for a, b in zip(*np.where(np.triu(D <= NEAR_DUP_HAMMING, k=1))):
        ga, gb = find(df.group.iat[idx[a]]), find(df.group.iat[idx[b]])
        if ga != gb:
            parent[gb] = ga
            near_pairs.append((df.path.iat[idx[a]], df.path.iat[idx[b]], int(D[a, b])))
df['group'] = df.group.map(find)
print(f'Near-duplicate merges: {len(near_pairs)}')

# --- split groups so that each emotion gets ~70/15/15 of its IMAGES (groups are never cut) ---
G = df.groupby('group').agg(label=('label', 'first'), n=('path', 'size'), n_orig=('is_original', 'sum'))
G['has_source'] = G.n_orig > 0
G['w'] = G.n if USE_ALL_PHOTOS else G.n_orig          # images the group contributes to val/test
rng = random.Random(SEED)
assign, split_share = {}, {}
for e in EMOTIONS:
    Ge = G[G.label == e]
    pool = Ge[Ge.has_source | USE_ALL_PHOTOS]
    for g in Ge.index.difference(pool.index):
        assign[g] = 'train'                              # no real source photo -> training only (strict mode)
    order = list(pool.index)
    rng.shuffle(order)                                   # random order among groups of equal size
    order.sort(key=lambda g: -pool.w[g])                 # place big groups first, small ones fill the gaps
    total, cur = pool.w.sum(), {s: 0 for s in SPLIT}
    for g in order:
        s = max(SPLIT, key=lambda s: SPLIT[s] * total - cur[s])
        assign[g] = s
        cur[s] += pool.w[g]
    split_share[e] = {s: round(float(cur[s]) / max(float(total), 1.0), 3) for s in SPLIT}
print('Share of images per split:', split_share)
df['split'] = df.group.map(assign)
df['aug_only_group'] = ~df.group.map(G.has_source)

df['used'] = (df.split == 'train') | df.is_original | USE_ALL_PHOTOS
for _, r in df[~df.used].iterrows():
    excluded.append((r.path, r.label, f'augmented variant of a {r.split} source photo (kept out to avoid leakage)'))
if EVAL_ONE_PER_GROUP:
    ev = df[df.used & df.split.isin(['val', 'test'])]
    drop = ev[ev.duplicated('group', keep='first')].index
    for i in drop:
        excluded.append((df.path[i], df.label[i], 'extra photo of an evaluation group (EVAL_ONE_PER_GROUP)'))
    df.loc[drop, 'used'] = False

lab2id = {e: i for i, e in enumerate(EMOTIONS)}
df['y'] = df.label.map(lab2id)
train_df = df[(df.split == 'train') & df.used].reset_index(drop=True)
val_df   = df[(df.split == 'val') & df.used].reset_index(drop=True)
test_df  = df[(df.split == 'test') & df.used].reset_index(drop=True)

### Split checks and dataset summary

In [ ]:
def used_split(s):
    return df[(df.split == s) & df.used]

checks = []
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    A, B = used_split(a), used_split(b)
    checks.append((f'source groups shared by {a} and {b}', len(set(A.group) & set(B.group))))
    checks.append((f'identical files shared by {a} and {b}', len(set(A.md5) & set(B.md5))))
cross_class = {}
for s in ['val', 'test']:
    S = used_split(s)
    D = hamming_flip(S.path.values, train_df.path.values)
    close = D <= NEAR_DUP_HAMMING
    same = np.array(S.label.tolist())[:, None] == np.array(train_df.label.tolist())[None, :]
    checks.append((f'{s} images with a near-duplicate (<= {NEAR_DUP_HAMMING} bits) of the SAME emotion in train', int((close & same).any(1).sum())))
    cross_class[s] = int((close & ~same).any(1).sum())
check_df = pd.DataFrame(checks, columns=['check', 'count'])
check_df['status'] = np.where(check_df['count'] == 0, 'OK', 'FAIL')
display(check_df)
assert (check_df['count'] == 0).all(), 'Leakage check failed - see the table above'
print('Info (not leakage): images that look very similar to a training photo with a DIFFERENT emotion '
      '(usually the same person and background):', cross_class)

summary = []
for s in ['train', 'val', 'test']:
    for e in EMOTIONS:
        d = df[(df.split == s) & (df.label == e)]
        u = d[d.used]
        summary.append({'split': s, 'emotion': e,
                        'source groups': d[~d.aug_only_group].group.nunique(),
                        'aug-only groups': d[d.aug_only_group].group.nunique(),
                        'real source photos used': int(u.is_original.sum()),
                        'augmented variants used': int((~u.is_original).sum()),
                        'images used': len(u)})
summary = pd.DataFrame(summary)
display(summary)
summary.to_csv(f'{RESULTS_DIR}/dataset_summary.csv', index=False)

ex = pd.DataFrame(excluded, columns=['path', 'label', 'reason'])
print(f'\nExcluded files: {len(ex)}')
if len(ex):
    display(ex.groupby(['reason', 'label']).size().unstack(fill_value=0))
ex.to_csv(f'{RESULTS_DIR}/excluded.csv', index=False)

aug_only = G[~G.has_source]
print(f'\nSource groups without a real source photo: {len(aug_only)}' + ('' if USE_ALL_PHOTOS else ' (training only)'))
if len(aug_only):
    print('  e.g.', list(aug_only.index[:8]))
big = G.sort_values('n', ascending=False).head(3)
print('Largest groups:', dict(zip(big.index, big.n)))

for s, d in [('val', val_df), ('test', test_df)]:
    small = {e: int((d.label == e).sum()) for e in EMOTIONS}
    if min(small.values()) < 5:
        print(f'WARNING: {s} set has fewer than 5 photos for some emotion {small} - metrics will be very unstable.')

def rel(p):
    return os.path.relpath(p, DATA_DIR)
src_tab = df[df.is_original & df.used].assign(source_key=lambda d: d.source, file=lambda d: d.path.map(rel))
src_tab[src_tab.split == 'train'][['label', 'source_key', 'file']].to_csv(f'{RESULTS_DIR}/train_sources.csv', index=False)
src_tab[src_tab.split == 'test'][['label', 'source_key', 'file']].to_csv(f'{RESULTS_DIR}/test_sources.csv', index=False)
df.assign(file=df.path.map(rel)).drop(columns=['path'])[['file', 'label', 'source', 'group', 'is_original', 'aug_only_group', 'split', 'used']].to_csv(f'{RESULTS_DIR}/splits.csv', index=False)
print(f'\nSaved splits.csv, excluded.csv, train_sources.csv, test_sources.csv to {RESULTS_DIR}')

## 2. PyTorch data pipeline

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T, models as tvm
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED)
ENV = f"{platform.system()} | torch {torch.__version__} | " + (torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')
print('Device:', ENV)

IMAGENET_NORM = ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
VIT_NORM      = ([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])

def make_tf(norm, augment):
    t = ([T.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0), ratio=(0.9, 1.1)), T.RandomHorizontalFlip(),
          T.RandomRotation(15), T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1)]
         if augment else [T.Resize((IMG_SIZE, IMG_SIZE))])
    return T.Compose(t + [T.ToTensor(), T.Normalize(*norm)])

class FaceDS(Dataset):
    def __init__(self, d, tf):
        self.p, self.y, self.tf = list(d.path), list(d.y), tf
    def __len__(self):
        return len(self.p)
    def __getitem__(self, i):
        return self.tf(Image.open(self.p[i]).convert('RGB')), self.y[i]

def loader(d, tf, shuffle=False):
    return DataLoader(FaceDS(d, tf), batch_size=BATCH, shuffle=shuffle, num_workers=2,
                      pin_memory=(device == 'cuda'), generator=torch.Generator().manual_seed(SEED))

counts = np.bincount(train_df.y, minlength=len(EMOTIONS))
class_w = torch.tensor(len(train_df) / (len(EMOTIONS) * np.maximum(counts, 1)), dtype=torch.float32)
print('Training images per class:', dict(zip(EMOTIONS, counts)), '| class weights:', class_w.numpy().round(2))

## 3. Shared training and evaluation code

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, f1_score

results, histories = {}, {}

def fit(model, train_dl, val_dl, epochs, param_groups, name, freeze_bn=False):
    model.to(device)
    opt = torch.optim.AdamW(param_groups, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=2)
    crit = nn.CrossEntropyLoss(weight=class_w.to(device))
    scaler = torch.amp.GradScaler('cuda', enabled=(device == 'cuda'))
    hist, best, bad = defaultdict(list), (np.inf, None), 0
    for ep in range(epochs):
        model.train()
        if freeze_bn:
            for m in model.modules():
                if isinstance(m, nn.BatchNorm2d):
                    m.eval()
        tl = tc = tn = 0
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device, enabled=(device == 'cuda')):
                out = model(x)
                loss = crit(out, y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            tl += loss.item() * len(y); tc += (out.argmax(1) == y).sum().item(); tn += len(y)
        vl, va = val_loss(model, val_dl, crit)
        sched.step(vl)
        for k, v in [('loss', tl / tn), ('accuracy', tc / tn), ('val_loss', vl), ('val_accuracy', va)]:
            hist[k].append(v)
        print(f'{name} | epoch {ep + 1:2d}/{epochs} | loss {tl / tn:.3f} acc {tc / tn:.3f} | val_loss {vl:.3f} val_acc {va:.3f}')
        if vl < best[0] - 1e-4:
            best, bad = (vl, {k: v.detach().clone() for k, v in model.state_dict().items()}), 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f'{name} | early stopping, best val_loss {best[0]:.3f}')
                break
    model.load_state_dict(best[1])
    return dict(hist)

@torch.no_grad()
def val_loss(model, dl, crit):
    model.eval()
    tl = tc = tn = 0
    for x, y in dl:
        x, y = x.to(device), y.to(device)
        out = model(x)
        tl += crit(out, y).item() * len(y); tc += (out.argmax(1) == y).sum().item(); tn += len(y)
    return tl / tn, tc / tn

@torch.no_grad()
def predict(model, dl):
    model.eval()
    return np.concatenate([torch.softmax(model(x.to(device)).float(), 1).cpu().numpy() for x, _ in dl])

def evaluate(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    results[name] = {'Accuracy': acc, 'Precision (macro)': p, 'Recall (macro)': r, 'F1 (macro)': f1}
    rep = classification_report(y_true, y_pred, labels=range(len(EMOTIONS)), target_names=EMOTIONS, digits=3, zero_division=0)
    print(f'===== {name} =====\n{rep}')
    tag = re.sub(r'[^A-Za-z0-9]+', '_', name)
    open(f'{RESULTS_DIR}/report_{tag}.txt', 'w').write(rep)
    cm = confusion_matrix(y_true, y_pred, labels=range(len(EMOTIONS)))
    plt.figure(figsize=(4.2, 3.6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=EMOTIONS, yticklabels=EMOTIONS, cbar=False)
    plt.xlabel('Predicted'); plt.ylabel('True'); plt.title(name)
    plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/cm_{tag}.png', dpi=150); plt.show()

def plot_history(hist, name, split_at=None):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    for ax, m in zip(axes, ['accuracy', 'loss']):
        ax.plot(hist[m], label='train'); ax.plot(hist['val_' + m], label='val')
        if split_at:
            ax.axvline(split_at - 0.5, color='gray', ls='--', label='fine-tuning starts')
        ax.set_title(f'{name} - {m}'); ax.set_xlabel('epoch'); ax.legend()
    tag = re.sub(r'[^A-Za-z0-9]+', '_', name)
    plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/curves_{tag}.png', dpi=150); plt.show()

def merge_hist(h1, h2):
    return {k: h1[k] + h2[k] for k in h1}

y_test = test_df.y.values

## 4. A1 baselines re-run on the clean split
The A1 numbers were measured on a split where augmented variants of one photo could appear in both training and test.
To compare fairly, the same three baselines are trained again here on the leak-free split (PyTorch re-implementation of the A1 recipes).

In [ ]:
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def hog_features(paths, flip=False):
    out = []
    for p in paths:
        g = cv2.resize(cv2.imread(p, cv2.IMREAD_GRAYSCALE), (96, 96))
        if flip:
            g = cv2.flip(g, 1)
        out.append(hog(g, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2)))
    return np.array(out)

X_tr = np.vstack([hog_features(train_df.path), hog_features(train_df.path, flip=True)])
y_tr = np.concatenate([train_df.y.values, train_df.y.values])
X_va, X_te = hog_features(val_df.path), hog_features(test_df.path)
best = (None, -1)
for C_ in [0.1, 1, 10, 100]:
    for gamma in ['scale', 1e-3, 1e-4]:
        m = make_pipeline(StandardScaler(), SVC(C=C_, gamma=gamma, kernel='rbf', class_weight='balanced'))
        m.fit(X_tr, y_tr)
        acc = accuracy_score(val_df.y, m.predict(X_va))
        if acc > best[1]:
            best = ((C_, gamma, m), acc)
(C_, gamma, svm_model), svm_val = best
print(f'Best SVM: C={C_}, gamma={gamma}, val accuracy={svm_val:.3f}')
evaluate('HOG + SVM', y_test, svm_model.predict(X_te))

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, n=len(EMOTIONS)):
        super().__init__()
        def blk(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.f = nn.Sequential(blk(3, 32), blk(32, 64), blk(64, 128), nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                               nn.Dropout(0.4), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, n))
    def forward(self, x):
        return self.f(x)

tr_cnn, va_cnn, te_cnn = (loader(train_df, make_tf(IMAGENET_NORM, True), True),
                          loader(val_df, make_tf(IMAGENET_NORM, False)), loader(test_df, make_tf(IMAGENET_NORM, False)))
cnn = SimpleCNN()
histories['Simple CNN'] = fit(cnn, tr_cnn, va_cnn, CNN_EPOCHS, [{'params': cnn.parameters(), 'lr': 1e-3}], 'Simple CNN')
plot_history(histories['Simple CNN'], 'Simple CNN')
evaluate('Simple CNN', y_test, predict(cnn, te_cnn).argmax(1))

In [ ]:
mnet = tvm.mobilenet_v2(weights=tvm.MobileNet_V2_Weights.IMAGENET1K_V1 if PRETRAINED else None)
mnet.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(1280, len(EMOTIONS)))
for p in mnet.features.parameters():
    p.requires_grad = False
h1 = fit(mnet, tr_cnn, va_cnn, MOBILENET_EPOCHS[0], [{'params': mnet.classifier.parameters(), 'lr': 1e-3}],
         'MobileNetV2 head', freeze_bn=True)
for p in mnet.features[14:].parameters():          # unfreeze the last inverted-residual blocks
    p.requires_grad = True
h2 = fit(mnet, tr_cnn, va_cnn, MOBILENET_EPOCHS[1],
         [{'params': [p for p in mnet.parameters() if p.requires_grad], 'lr': 1e-5}], 'MobileNetV2 fine-tune', freeze_bn=True)
histories['MobileNetV2'] = merge_hist(h1, h2)
plot_history(histories['MobileNetV2'], 'MobileNetV2', split_at=len(h1['loss']))
evaluate('MobileNetV2', y_test, predict(mnet, te_cnn).argmax(1))

## 5. SOTA classifier: Vision Transformer (ViT-B/16) + ablation
ViT splits the 224×224 face into 196 patches of 16×16 pixels, embeds each patch, and relates all patches with 12 layers of multi-head self-attention.
The checkpoint is pretrained on ImageNet-21k; the whole network is fine-tuned (backbone lr 2e-5, new head lr 1e-3).

**Ablation** (same split, same epochs, only one factor changed):
* `ViT (no augmentation)` — resize + normalise only
* `ViT (augmentation)` — random crop, flip, rotation ±15°, colour jitter
* `ViT (augmentation + ArtAug)` — additionally trains on ArtAug-enhanced versions of **training** photos (runs only if `A2/artaug` exists)

In [ ]:
from transformers import ViTForImageClassification

class ViTWrap(nn.Module):
    def __init__(self, hf):
        super().__init__()
        self.hf = hf
    def forward(self, x):
        return self.hf(pixel_values=x).logits

def build_vit():
    hf = ViTForImageClassification.from_pretrained(
        VIT_CHECKPOINT, num_labels=len(EMOTIONS), id2label=dict(enumerate(EMOTIONS)), label2id=lab2id,
        attn_implementation='eager', ignore_mismatched_sizes=True)
    return ViTWrap(hf)

# ---- optional ArtAug images (must come from TRAINING source photos) ----
art_rows = []
if os.path.isdir(ARTAUG_DIR):
    for e in EMOTIONS:
        for p in sorted(glob.glob(os.path.join(ARTAUG_DIR, e, '*'))):
            if not p.lower().endswith(IMG_EXTS):
                continue
            src = os.path.splitext(os.path.basename(p))[0].split('__')[0].lower()
            g = f'{e}::{source_key(src)}'
            g = find(g) if g in parent else None
            if g is None or assign.get(g) != 'train':
                excluded.append((p, e, 'ArtAug image whose source is not a training photo'))
                continue
            art_rows.append({'path': p, 'label': e, 'y': lab2id[e], 'group': g})
art_df = pd.DataFrame(art_rows)
print(f'ArtAug images accepted: {len(art_df)}' + ('' if len(art_df) else ' (folder missing or empty - ArtAug ablation is skipped)'))

va_vit, te_vit = loader(val_df, make_tf(VIT_NORM, False)), loader(test_df, make_tf(VIT_NORM, False))
vit_models, vit_val_f1 = {}, {}

def train_vit(name, augment=True, extra=None):
    m = build_vit()
    d = pd.concat([train_df, extra], ignore_index=True) if extra is not None and len(extra) else train_df
    head_ids = {id(p) for p in m.hf.classifier.parameters()}
    groups = [{'params': [p for p in m.parameters() if id(p) not in head_ids], 'lr': 2e-5},
              {'params': list(m.hf.classifier.parameters()), 'lr': 1e-3}]
    histories[name] = fit(m, loader(d, make_tf(VIT_NORM, augment), True), va_vit, VIT_EPOCHS, groups, name)
    plot_history(histories[name], name)
    evaluate(name, y_test, predict(m, te_vit).argmax(1))
    vit_val_f1[name] = f1_score(val_df.y, predict(m, va_vit).argmax(1), average='macro')
    vit_models[name] = m.cpu()
    if device == 'cuda':
        torch.cuda.empty_cache()

train_vit('ViT (no augmentation)', augment=False)
train_vit('ViT (augmentation)', augment=True)
if len(art_df):
    train_vit('ViT (augmentation + ArtAug)', augment=True, extra=art_df)

BEST_VIT = max(vit_val_f1, key=vit_val_f1.get)     # chosen on VALIDATION, not test
best_vit = vit_models[BEST_VIT].to(device).eval()
print('Validation macro-F1:', {k: round(v, 3) for k, v in vit_val_f1.items()}, '-> best:', BEST_VIT)

## 6. Results: baselines vs ViT

In [ ]:
table = pd.DataFrame(results).T.round(3)
display(table)
table.to_csv(f'{RESULTS_DIR}/model_comparison.csv')

ax = table.plot(kind='bar', figsize=(12, 4), ylim=(0, 1), rot=15)
ax.set_ylabel('score (test set)'); ax.set_title('Baselines vs ViT (clean split)'); ax.legend(loc='lower right', fontsize=8)
for c in ax.containers:
    ax.bar_label(c, fmt='%.2f', fontsize=6)
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/model_comparison.png', dpi=150); plt.show()

A1_REPORTED = pd.DataFrame({'Accuracy': [0.889, 0.333, 0.600], 'F1 (macro)': [0.889, 0.338, 0.582]},
                           index=['HOG + SVM', 'Simple CNN', 'MobileNetV2'])
print('A1 numbers as reported (old split, for reference only - not comparable):')
display(A1_REPORTED)

## 7. Explainability: attention rollout
Attention rollout (Abnar & Zuidema, 2020) multiplies the head-averaged attention matrices of all layers (plus the residual identity)
and reads how much the `[CLS]` token attends to each patch — a 14×14 map of where the ViT "looks".

In [ ]:
@torch.no_grad()
def attention_rollout(vit, x):
    out = vit.hf(pixel_values=x, output_attentions=True)
    att = torch.stack(out.attentions).float().mean(2)            # layers x batch x tokens x tokens
    a = att + torch.eye(att.size(-1), device=att.device)
    a = a / a.sum(-1, keepdim=True)
    roll = a[0]
    for l in range(1, a.size(0)):
        roll = a[l] @ roll
    m = roll[:, 0, 1:]
    s = int(m.size(-1) ** 0.5)
    m = m.reshape(-1, s, s)
    m = m / m.amax((1, 2), keepdim=True)
    return m.cpu().numpy(), torch.softmax(out.logits.float(), -1).cpu().numpy()

def overlay(rgb, mask, alpha=0.45):
    heat = cv2.applyColorMap((cv2.resize(mask, rgb.shape[1::-1]) * 255).astype(np.uint8), cv2.COLORMAP_JET)
    return (rgb * (1 - alpha) + heat[:, :, ::-1] * alpha).astype(np.uint8)

tf_eval = make_tf(VIT_NORM, False)
picks = [i for e in range(len(EMOTIONS)) for i in np.where(y_test == e)[0][:2]]
fig, axes = plt.subplots(2, len(picks), figsize=(2.6 * len(picks), 5.6), squeeze=False)
for c, i in enumerate(picks):
    pil = Image.open(test_df.path[i]).convert('RGB')
    mask, prob = attention_rollout(best_vit, tf_eval(pil)[None].to(device))
    rgb = np.array(pil.resize((IMG_SIZE, IMG_SIZE)))
    axes[0, c].imshow(rgb); axes[0, c].set_title(f'true: {EMOTIONS[y_test[i]]}', fontsize=9)
    axes[1, c].imshow(overlay(rgb, mask[0])); axes[1, c].set_title(f'pred: {EMOTIONS[prob[0].argmax()]} ({prob[0].max():.2f})', fontsize=9)
for ax in axes.flat:
    ax.axis('off')
plt.suptitle(f'Attention rollout - {BEST_VIT}'); plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/attention_rollout.png', dpi=150); plt.show()

## 8. Latency (batch of 1, average of 30 runs after warm-up)

In [ ]:
def latency_ms(model, dev, runs=30):
    model = model.to(dev).eval()
    x = tf_eval(Image.open(test_df.path[0]).convert('RGB'))[None].to(dev)
    with torch.no_grad():
        for _ in range(5):
            model(x)
        if dev == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(runs):
            model(x)
        if dev == 'cuda': torch.cuda.synchronize()
    return (time.perf_counter() - t0) / runs * 1000

lat = []
for name, m in [('Simple CNN', cnn), ('MobileNetV2', mnet), (BEST_VIT, best_vit)]:
    n_params = sum(p.numel() for p in m.parameters()) / 1e6
    row = {'model': name, 'params (M)': round(n_params, 2), 'CPU ms/image': round(latency_ms(m, 'cpu'), 1)}
    if device == 'cuda':
        row['GPU ms/image'] = round(latency_ms(m, 'cuda'), 1)
    lat.append(row)
best_vit = best_vit.to(device)
lat = pd.DataFrame(lat)
display(lat); print('Environment:', ENV, '| CPU:', platform.processor() or platform.machine())
lat.to_csv(f'{RESULTS_DIR}/latency.csv', index=False)

## 9. Evaluation under animation (VASA-1 concept → LivePortrait / SadTalker clips)
Each clip in `A2/clips/<emotion>/` must be generated from a **test** source photo and named `<source_key>__anything.mp4`
(source keys are in `test_sources.csv`). The clip inherits the emotion of its source photo.

* **Stills macro-F1** — the ViT on the source photos of the clips
* **Frames macro-F1** — the ViT on every `FRAME_STEP`-th frame of the clips
* **Δmacro-F1 = frames − stills**, plus per-clip *consistency* (share of frames that agree with the clip's majority prediction)

All clips are synthetic media and are labelled as such in the report and the app.

In [ ]:
VID_EXTS = ('.mp4', '.mov', '.webm', '.avi', '.mkv')
frame_rows, clip_rows = [], []
if os.path.isdir(CLIPS_DIR):
    for e in EMOTIONS:
        for v in sorted(glob.glob(os.path.join(CLIPS_DIR, e, '*'))):
            if not v.lower().endswith(VID_EXTS):
                continue
            src = os.path.splitext(os.path.basename(v))[0].split('__')[0].lower()
            g = f'{e}::{source_key(src)}'
            g = find(g) if g in parent else None
            if g is None or assign.get(g) != 'test':
                print(f'skipped {os.path.basename(v)}: source is not a test photo ({assign.get(g, "unknown")})')
                continue
            cap, k, frames = cv2.VideoCapture(v), 0, []
            while True:
                ok, fr = cap.read()
                if not ok:
                    break
                if k % FRAME_STEP == 0:
                    frames.append(tf_eval(Image.fromarray(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))))
                k += 1
            cap.release()
            if not frames:
                continue
            with torch.no_grad():
                probs = np.concatenate([torch.softmax(best_vit(torch.stack(frames[i:i + 32]).to(device)).float(), 1).cpu().numpy()
                                        for i in range(0, len(frames), 32)])
            pred = probs.argmax(1)
            majority = np.bincount(pred, minlength=len(EMOTIONS)).argmax()
            clip = os.path.basename(v)
            clip_rows.append({'clip': clip, 'emotion': e, 'group': g, 'frames': len(pred),
                              'frame accuracy': (pred == lab2id[e]).mean(), 'majority': EMOTIONS[majority],
                              'consistency': (pred == majority).mean()})
            for j, (pp, pr) in enumerate(zip(pred, probs)):
                frame_rows.append({'clip': clip, 'frame': j * FRAME_STEP, 'y': lab2id[e], 'pred': pp, **dict(zip(EMOTIONS, pr))})

if clip_rows:
    clips_df, frames_df = pd.DataFrame(clip_rows), pd.DataFrame(frame_rows)
    display(clips_df.round(3))
    stills = test_df[test_df.group.isin(clips_df.group)]
    still_pred = predict(best_vit, loader(stills, tf_eval)).argmax(1)
    labels = sorted(set(stills.y) | set(frames_df.y))
    f1_still = f1_score(stills.y, still_pred, labels=labels, average='macro', zero_division=0)
    f1_frame = f1_score(frames_df.y, frames_df.pred, labels=labels, average='macro', zero_division=0)
    anim = pd.DataFrame([{'clips': len(clips_df), 'source stills': len(stills), 'frames': len(frames_df),
                          'macro-F1 stills': f1_still, 'macro-F1 frames': f1_frame, 'delta macro-F1': f1_frame - f1_still,
                          'mean consistency': clips_df.consistency.mean()}]).round(3)
    display(anim)
    anim.to_csv(f'{RESULTS_DIR}/animation_eval.csv', index=False)
    clips_df.to_csv(f'{RESULTS_DIR}/animation_clips.csv', index=False)

    show = clips_df.groupby('emotion').head(1)['clip'].tolist()
    fig, axes = plt.subplots(len(show), 1, figsize=(10, 2.6 * len(show)), squeeze=False)
    for ax, c in zip(axes[:, 0], show):
        f = frames_df[frames_df['clip'] == c]
        for e in EMOTIONS:
            ax.plot(f.frame, f[e], label=e)
        r = clips_df[clips_df['clip'] == c].iloc[0]
        ax.set_ylim(0, 1); ax.set_ylabel('probability')
        ax.set_title(f'{c} (source: {r.emotion}) - consistency {r.consistency:.2f} [SYNTHETIC CLIP]', fontsize=9)
    axes[-1, 0].set_xlabel('frame'); axes[0, 0].legend(loc='upper right', fontsize=8)
    plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/emotion_timeline.png', dpi=150); plt.show()
else:
    print('No clips found in', CLIPS_DIR, '- generate them from test_sources.csv, then re-run this cell.')

## 10. Export the model for the app

In [ ]:
EXPORT = f'{RESULTS_DIR}/vit_emotion'
best_vit.hf.save_pretrained(EXPORT)
json.dump({'labels': EMOTIONS, 'img_size': IMG_SIZE, 'mean': VIT_NORM[0], 'std': VIT_NORM[1], 'variant': BEST_VIT},
          open(f'{EXPORT}/app_config.json', 'w'), indent=2)
print('Saved to', EXPORT, os.listdir(EXPORT))

## 11. Quick demo (Gradio)
Upload a photo or use the webcam → emotion probabilities → attention heatmap → latency and environment.
`share=True` gives a temporary public link (valid while this Colab session runs) — good for testing; the permanent app is deployed separately.

In [ ]:
!pip -q install gradio
import gradio as gr

def predict_ui(img):
    if img is None:
        return None, None, ''
    pil = Image.fromarray(img).convert('RGB')
    x = tf_eval(pil)[None].to(device)
    if device == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    mask, prob = attention_rollout(best_vit, x)
    if device == 'cuda': torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) * 1000
    heat = overlay(np.array(pil.resize((IMG_SIZE, IMG_SIZE))), mask[0])
    return {e: float(prob[0][i]) for i, e in enumerate(EMOTIONS)}, heat, f'{ms:.0f} ms (prediction + explanation) | {ENV}'

gr.Interface(
    fn=predict_ui,
    inputs=gr.Image(sources=['upload', 'webcam'], type='numpy', label='Face photo'),
    outputs=[gr.Label(num_top_classes=3, label='Emotion'), gr.Image(label='Attention rollout'), gr.Textbox(label='Latency / environment')],
    title='Facial emotion recognition (ViT) - happy / sad / neutral',
    description='Images are processed only for this prediction and are not stored. Use only photos of people who agreed to it.',
).launch(share=True)